In [0]:
source_path = (
    "/Volumes/real_time_catalogue/"
    "bronze_realtime/"
    "landing_events/"
    "transactions"
)

schema_path = (
    "/Volumes/real_time_catalogue/"
    "bronze_realtime/"
    "landing_events/"
    "_schemas/transactions"
)

checkpoint_path = (
    "/Volumes/real_time_catalogue/"
    "bronze_realtime/"
    "landing_events/"
    "_checkpoints/transactions"
)

bronze_table = (
    "real_time_catalogue."
    "bronze_realtime."
    "transactions"
)

print("Source :", source_path)
print("Table Bronze :", bronze_table)

Source : /Volumes/real_time_catalogue/bronze_realtime/landing_events/transactions
Table Bronze : real_time_catalogue.bronze_realtime.transactions


In [0]:
from pyspark.sql.functions import col, current_timestamp

bronze_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .load(source_path)
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_file", col("_metadata.file_path"))
)

In [0]:
bronze_stream.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- source_type: string (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- transaction_date: string (nullable = true)
 |-- transaction_id: string (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = false)
 |-- source_file: string (nullable = false)



In [0]:
bronze_query = (
    bronze_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(bronze_table)
)

bronze_query.awaitTermination()

print("Chargement Bronze terminé.")

Chargement Bronze terminé.


In [0]:
%sql
SELECT *
FROM real_time_catalogue.bronze_realtime.transactions
ORDER BY ingestion_timestamp DESC;

customer_id,event_type,product_id,quantity,source_type,total_amount,transaction_date,transaction_id,unit_price,_rescued_data,ingestion_timestamp,source_file
C06233,transaction,P0564,1,realtime,68.18,2026-09-07T12:55:22.592108+00:00,RT-6626AEBE3AF6,68.18,null,2026-09-07T13:05:27.315Z,/Volumes/real_time_catalogue/bronze_realtime/landing_events/transactions/transaction_1788785722592_8.json
C02058,transaction,P0360,2,realtime,349.38,2026-09-07T12:55:27.542481+00:00,RT-E9699EC1C14E,174.69,null,2026-09-07T13:05:27.315Z,/Volumes/real_time_catalogue/bronze_realtime/landing_events/transactions/transaction_1788785727542_10.json
C09621,transaction,P0108,4,realtime,509.8,2026-09-07T12:55:25.084912+00:00,RT-E9CEA38E3006,127.45,null,2026-09-07T13:05:27.315Z,/Volumes/real_time_catalogue/bronze_realtime/landing_events/transactions/transaction_1788785725084_9.json
C05193,transaction,P0209,4,realtime,308.2,2026-09-07T12:55:14.976556+00:00,RT-BA87BA5D15AD,77.05,null,2026-09-07T13:05:27.315Z,/Volumes/real_time_catalogue/bronze_realtime/landing_events/transactions/transaction_1788785714976_5.json
C04791,transaction,P0454,3,realtime,1088.16,2026-09-07T12:55:07.349433+00:00,RT-89CC914D0E0E,362.72,null,2026-09-07T13:05:27.315Z,/Volumes/real_time_catalogue/bronze_realtime/landing_events/transactions/transaction_1788785707349_2.json
C09742,transaction,P0057,1,realtime,287.42,2026-09-07T12:55:20.095235+00:00,RT-F3BE88BB2F0E,287.42,null,2026-09-07T13:05:27.315Z,/Volumes/real_time_catalogue/bronze_realtime/landing_events/transactions/transaction_1788785720095_7.json
C03895,transaction,P0746,4,realtime,1133.72,2026-09-07T12:55:04.594124+00:00,RT-FC80185510BE,283.43,null,2026-09-07T13:05:27.315Z,/Volumes/real_time_catalogue/bronze_realtime/landing_events/transactions/transaction_1788785704594_1.json
C02638,transaction,P0462,2,realtime,283.22,2026-09-07T12:55:17.610241+00:00,RT-0975346145A1,141.61,null,2026-09-07T13:05:27.315Z,/Volumes/real_time_catalogue/bronze_realtime/landing_events/transactions/transaction_1788785717610_6.json
C07618,transaction,P0924,5,realtime,1627.45,2026-09-07T12:55:09.931810+00:00,RT-5C7FF6B821A5,325.49,null,2026-09-07T13:05:27.315Z,/Volumes/real_time_catalogue/bronze_realtime/landing_events/transactions/transaction_1788785709931_3.json
C07160,transaction,P0308,4,realtime,186.8,2026-09-07T12:55:12.480349+00:00,RT-FE607FE30AE2,46.7,null,2026-09-07T13:05:27.315Z,/Volumes/real_time_catalogue/bronze_realtime/landing_events/transactions/transaction_1788785712480_4.json


In [0]:
%sql
SELECT COUNT(*) AS number_transactions
FROM real_time_catalogue.bronze_realtime.transactions;

number_transactions
10
